In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import gc
import io
import os
from itertools import combinations

from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

pd.reset_option('display.float_format')
pd.set_option('display.max_colwidth', None)

from config import ROOT, prev_num_aggregations  # lib này được khởi tạo ban đầu dự án

import helpers.view as view
import helpers.EDA as EDA
import modules.utils as utils

importlib.reload(view)
importlib.reload(EDA)
importlib.reload(utils)

from helpers.cache_clear import cache_clear

get_pickle = utils.get_pickle
get_pickles = utils.get_pickles

In [2]:
stats = ['min', 'mean', 'max', 'std']
stats_sum = ['min', 'mean', 'max', 'std', 'sum']
prev_num_aggregations = {
    "AMT_ANNUITY": stats,
    "AMT_APPLICATION": stats,
    "AMT_CREDIT": stats,
    "AMT_DOWN_PAYMENT": stats,
    "AMT_GOODS_PRICE": stats,
    "HOUR_APPR_PROCESS_START": stats,
    "FLAG_LAST_APPL_PER_CONTRACT": stats, # FLAG_LAST_APPL_PER_CONTRACT nhị phân hóa từ extract
    "NFLAG_LAST_APPL_IN_DAY": stats,
    "RATE_DOWN_PAYMENT": stats,
    "RATE_INTEREST_PRIMARY": stats,
    "RATE_INTEREST_PRIVILEGED": stats,
    "SELLERPLACE_AREA": stats,
    "CNT_PAYMENT": stats,
    
    "DAYS_FIRST_DRAWING-s-DAYS_DECISIONS": stats,
    "DAYS_FIRST_DUE-s-DAYS_DECISIONS": stats,
    "DAYS_LAST_DUE_1ST_VERSION-s-DAYS_DECISIONS": stats,
    "DAYS_LAST_DUE-s-DAYS_DECISIONS": stats,
    "DAYS_TERMINATION-s-DAYS_DECISIONS": stats,
    "DAYS_FIRST_DUE-s-DAYS_FIRST_DRAWING": stats,
    "DAYS_LAST_DUE_1ST_VERSION-s-DAYS_FIRST_DRAWING": stats,
    "DAYS_LAST_DUE-s-DAYS_FIRST_DRAWING": stats,
    "DAYS_TERMINATION-s-DAYS_FIRST_DRAWING": stats,
    "DAYS_LAST_DUE_1ST_VERSION-s-DAYS_FIRST_DUE": stats,
    "DAYS_LAST_DUE-s-DAYS_FIRST_DUE": stats,
    "DAYS_TERMINATION-s-DAYS_FIRST_DUE": stats,
    "DAYS_LAST_DUE-s-DAYS_LAST_DUE_1ST_VERSION": stats,
    "DAYS_TERMINATION-s-DAYS_LAST_DUE_1ST_VERSION": stats,
    "DAYS_TERMINATION-s-DAYS_LAST_DUE": stats,
    "total_debt": stats,
    "AMT_GOODS_PRICE-d-total_debt": stats,
    "AMT_CREDIT-d-total_debt": stats,
    "AMT_CREDIT-d-AMT_ANNUITY": stats,
    "AMT_GOODS_PRICE-d-AMT_ANNUITY": stats,
    "AMT_CREDIT-d-AMT_APPLICATION": stats,
    "AMT_GOODS_PRICE-d-AMT_CREDIT": stats,
    "AMT_DOWN_PAYMENT-d-AMT_GOODS_PRICE": stats,
    "AMT_ANNUITY-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_APPLICATION-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_CREDIT-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_ANNUITY-s-app_AMT_INCOME_TOTAL": stats,
    "AMT_APPLICATION-s-app_AMT_INCOME_TOTAL": stats,
    "AMT_CREDIT-s-app_AMT_INCOME_TOTAL": stats,
    "AMT_GOODS_PRICE-s-app_AMT_INCOME_TOTAL": stats,
    "AMT_ANNUITY-d-app_AMT_CREDIT": stats,
    "AMT_APPLICATION-d-app_AMT_CREDIT": stats,
    "AMT_CREDIT-d-app_AMT_CREDIT": stats,
    "AMT_GOODS_PRICE-d-app_AMT_CREDIT": stats,
    "AMT_ANNUITY-s-app_AMT_CREDIT": stats,
    "AMT_APPLICATION-s-app_AMT_CREDIT": stats,
    "AMT_CREDIT-s-app_AMT_CREDIT": stats,
    "AMT_GOODS_PRICE-s-app_AMT_CREDIT": stats,
    "AMT_ANNUITY-d-app_AMT_ANNUITY": stats,
    "AMT_APPLICATION-d-app_AMT_ANNUITY": stats,
    "AMT_CREDIT-d-app_AMT_ANNUITY": stats,
    "AMT_GOODS_PRICE-d-app_AMT_ANNUITY": stats,
    "AMT_ANNUITY-s-app_AMT_ANNUITY": stats,
    "AMT_APPLICATION-s-app_AMT_ANNUITY": stats,
    "AMT_CREDIT-s-app_AMT_ANNUITY": stats,
    "AMT_GOODS_PRICE-s-app_AMT_ANNUITY": stats,
    "AMT_ANNUITY-d-app_AMT_GOODS_PRICE": stats,
    "AMT_APPLICATION-d-app_AMT_GOODS_PRICE": stats,
    "AMT_CREDIT-d-app_AMT_GOODS_PRICE": stats,
    "AMT_GOODS_PRICE-d-app_AMT_GOODS_PRICE": stats,
    "AMT_ANNUITY-s-app_AMT_GOODS_PRICE": stats,
    "AMT_APPLICATION-s-app_AMT_GOODS_PRICE": stats,
    "AMT_CREDIT-s-app_AMT_GOODS_PRICE": stats,
    "AMT_GOODS_PRICE-s-app_AMT_GOODS_PRICE": stats,
    "AMT_ANNUITY-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_APPLICATION-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_CREDIT-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_GOODS_PRICE-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_ANNUITY-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_APPLICATION-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_CREDIT-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_GOODS_PRICE-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_ANNUITY-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_APPLICATION-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_CREDIT-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL": stats,
    "AMT_GOODS_PRICE-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL": stats,
    "interest_rate": stats,
    "total_debt_diff": stats,
    "total_debt_pctchange": stats,
    "AMT_GOODS_PRICE-d-total_debt_diff": stats,
    "AMT_GOODS_PRICE-d-total_debt_pctchange": stats,
    "AMT_CREDIT-d-total_debt_diff": stats,
    "AMT_CREDIT-d-total_debt_pctchange": stats,
    "AMT_CREDIT-d-AMT_ANNUITY_diff": stats,
    "AMT_CREDIT-d-AMT_ANNUITY_pctchange": stats,
    "AMT_GOODS_PRICE-d-AMT_ANNUITY_diff": stats,
    "AMT_GOODS_PRICE-d-AMT_ANNUITY_pctchange": stats,
    "AMT_CREDIT-d-AMT_APPLICATION_diff": stats,
    "AMT_CREDIT-d-AMT_APPLICATION_pctchange": stats,
    "AMT_GOODS_PRICE-d-AMT_CREDIT_diff": stats,
    "AMT_GOODS_PRICE-d-AMT_CREDIT_pctchange": stats,
    "AMT_DOWN_PAYMENT-d-AMT_GOODS_PRICE_diff": stats,
    "AMT_DOWN_PAYMENT-d-AMT_GOODS_PRICE_pctchange": stats,
    "AMT_ANNUITY-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_ANNUITY-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_APPLICATION-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_APPLICATION-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_CREDIT-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_CREDIT-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_ANNUITY-s-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_ANNUITY-s-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_APPLICATION-s-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_APPLICATION-s-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_CREDIT-s-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_CREDIT-s-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_GOODS_PRICE-s-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_GOODS_PRICE-s-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_ANNUITY-d-app_AMT_CREDIT_diff": stats,
    "AMT_ANNUITY-d-app_AMT_CREDIT_pctchange": stats,
    "AMT_APPLICATION-d-app_AMT_CREDIT_diff": stats,
    "AMT_APPLICATION-d-app_AMT_CREDIT_pctchange": stats,
    "AMT_CREDIT-d-app_AMT_CREDIT_diff": stats,
    "AMT_CREDIT-d-app_AMT_CREDIT_pctchange": stats,
    "AMT_GOODS_PRICE-d-app_AMT_CREDIT_diff": stats,
    "AMT_GOODS_PRICE-d-app_AMT_CREDIT_pctchange": stats,
    "AMT_ANNUITY-s-app_AMT_CREDIT_diff": stats,
    "AMT_ANNUITY-s-app_AMT_CREDIT_pctchange": stats,
    "AMT_APPLICATION-s-app_AMT_CREDIT_diff": stats,
    "AMT_APPLICATION-s-app_AMT_CREDIT_pctchange": stats,
    "AMT_CREDIT-s-app_AMT_CREDIT_diff": stats,
    "AMT_CREDIT-s-app_AMT_CREDIT_pctchange": stats,
    "AMT_GOODS_PRICE-s-app_AMT_CREDIT_diff": stats,
    "AMT_GOODS_PRICE-s-app_AMT_CREDIT_pctchange": stats,
    "AMT_ANNUITY-d-app_AMT_ANNUITY_diff": stats,
    "AMT_ANNUITY-d-app_AMT_ANNUITY_pctchange": stats,
    "AMT_APPLICATION-d-app_AMT_ANNUITY_diff": stats,
    "AMT_APPLICATION-d-app_AMT_ANNUITY_pctchange": stats,
    "AMT_CREDIT-d-app_AMT_ANNUITY_diff": stats,
    "AMT_CREDIT-d-app_AMT_ANNUITY_pctchange": stats,
    "AMT_GOODS_PRICE-d-app_AMT_ANNUITY_diff": stats,
    "AMT_GOODS_PRICE-d-app_AMT_ANNUITY_pctchange": stats,
    "AMT_ANNUITY-s-app_AMT_ANNUITY_diff": stats,
    "AMT_ANNUITY-s-app_AMT_ANNUITY_pctchange": stats,
    "AMT_APPLICATION-s-app_AMT_ANNUITY_diff": stats,
    "AMT_APPLICATION-s-app_AMT_ANNUITY_pctchange": stats,
    "AMT_CREDIT-s-app_AMT_ANNUITY_diff": stats,
    "AMT_CREDIT-s-app_AMT_ANNUITY_pctchange": stats,
    "AMT_GOODS_PRICE-s-app_AMT_ANNUITY_diff": stats,
    "AMT_GOODS_PRICE-s-app_AMT_ANNUITY_pctchange": stats,
    "AMT_ANNUITY-d-app_AMT_GOODS_PRICE_diff": stats,
    "AMT_ANNUITY-d-app_AMT_GOODS_PRICE_pctchange": stats,
    "AMT_APPLICATION-d-app_AMT_GOODS_PRICE_diff": stats,
    "AMT_APPLICATION-d-app_AMT_GOODS_PRICE_pctchange": stats,
    "AMT_CREDIT-d-app_AMT_GOODS_PRICE_diff": stats,
    "AMT_CREDIT-d-app_AMT_GOODS_PRICE_pctchange": stats,
    "AMT_GOODS_PRICE-d-app_AMT_GOODS_PRICE_diff": stats,
    "AMT_GOODS_PRICE-d-app_AMT_GOODS_PRICE_pctchange": stats,
    "AMT_ANNUITY-s-app_AMT_GOODS_PRICE_diff": stats,
    "AMT_ANNUITY-s-app_AMT_GOODS_PRICE_pctchange": stats,
    "AMT_APPLICATION-s-app_AMT_GOODS_PRICE_diff": stats,
    "AMT_APPLICATION-s-app_AMT_GOODS_PRICE_pctchange": stats,
    "AMT_CREDIT-s-app_AMT_GOODS_PRICE_diff": stats,
    "AMT_CREDIT-s-app_AMT_GOODS_PRICE_pctchange": stats,
    "AMT_GOODS_PRICE-s-app_AMT_GOODS_PRICE_diff": stats,
    "AMT_GOODS_PRICE-s-app_AMT_GOODS_PRICE_pctchange": stats,
    "AMT_ANNUITY-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_ANNUITY-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_APPLICATION-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_APPLICATION-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_CREDIT-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_CREDIT-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_GOODS_PRICE-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_GOODS_PRICE-s-app_AMT_CREDIT-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_ANNUITY-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_ANNUITY-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_APPLICATION-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_APPLICATION-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_CREDIT-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_CREDIT-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_GOODS_PRICE-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_GOODS_PRICE-s-app_AMT_ANNUITY-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_ANNUITY-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_ANNUITY-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_APPLICATION-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_APPLICATION-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_CREDIT-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_CREDIT-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "AMT_GOODS_PRICE-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL_diff": stats,
    "AMT_GOODS_PRICE-m-app_AMT_GOODS_PRICE-d-app_AMT_INCOME_TOTAL_pctchange": stats,
    "interest_rate_diff": stats,
    "interest_rate_pctchange": stats,
    "DAYS_DECISION-s-app_DAYS_BIRTH": stats,
    "DAYS_DECISION-d-app_DAYS_BIRTH": stats,
    "DAYS_DECISION-s-app_DAYS_EMPLOYED": stats,
    "DAYS_DECISION-d-app_DAYS_EMPLOYED": stats,
    "DAYS_DECISION-s-app_DAYS_REGISTRATION": stats,
    "DAYS_DECISION-d-app_DAYS_REGISTRATION": stats,
    "DAYS_DECISION-s-app_DAYS_ID_PUBLISH": stats,
    "DAYS_DECISION-d-app_DAYS_ID_PUBLISH": stats,
    "DAYS_DECISION-s-app_DAYS_LAST_PHONE_CHANGE": stats,
    "DAYS_DECISION-d-app_DAYS_LAST_PHONE_CHANGE": stats,
    "DAYS_FIRST_DRAWING-s-app_DAYS_BIRTH": stats,
    "DAYS_FIRST_DRAWING-d-app_DAYS_BIRTH": stats,
    "DAYS_FIRST_DRAWING-s-app_DAYS_EMPLOYED": stats,
    "DAYS_FIRST_DRAWING-d-app_DAYS_EMPLOYED": stats,
    "DAYS_FIRST_DRAWING-s-app_DAYS_REGISTRATION": stats,
    "DAYS_FIRST_DRAWING-d-app_DAYS_REGISTRATION": stats,
    "DAYS_FIRST_DRAWING-s-app_DAYS_ID_PUBLISH": stats,
    "DAYS_FIRST_DRAWING-d-app_DAYS_ID_PUBLISH": stats,
    "DAYS_FIRST_DRAWING-s-app_DAYS_LAST_PHONE_CHANGE": stats,
    "DAYS_FIRST_DRAWING-d-app_DAYS_LAST_PHONE_CHANGE": stats,
    "DAYS_FIRST_DUE-s-app_DAYS_BIRTH": stats,
    "DAYS_FIRST_DUE-d-app_DAYS_BIRTH": stats,
    "DAYS_FIRST_DUE-s-app_DAYS_EMPLOYED": stats,
    "DAYS_FIRST_DUE-d-app_DAYS_EMPLOYED": stats,
    "DAYS_FIRST_DUE-s-app_DAYS_REGISTRATION": stats,
    "DAYS_FIRST_DUE-d-app_DAYS_REGISTRATION": stats,
    "DAYS_FIRST_DUE-s-app_DAYS_ID_PUBLISH": stats,
    "DAYS_FIRST_DUE-d-app_DAYS_ID_PUBLISH": stats,
    "DAYS_FIRST_DUE-s-app_DAYS_LAST_PHONE_CHANGE": stats,
    "DAYS_FIRST_DUE-d-app_DAYS_LAST_PHONE_CHANGE": stats,
    "DAYS_LAST_DUE_1ST_VERSION-s-app_DAYS_BIRTH": stats,
    "DAYS_LAST_DUE_1ST_VERSION-d-app_DAYS_BIRTH": stats,
    "DAYS_LAST_DUE_1ST_VERSION-s-app_DAYS_EMPLOYED": stats,
    "DAYS_LAST_DUE_1ST_VERSION-d-app_DAYS_EMPLOYED": stats,
    "DAYS_LAST_DUE_1ST_VERSION-s-app_DAYS_REGISTRATION": stats,
    "DAYS_LAST_DUE_1ST_VERSION-d-app_DAYS_REGISTRATION": stats,
    "DAYS_LAST_DUE_1ST_VERSION-s-app_DAYS_ID_PUBLISH": stats,
    "DAYS_LAST_DUE_1ST_VERSION-d-app_DAYS_ID_PUBLISH": stats,
    "DAYS_LAST_DUE_1ST_VERSION-s-app_DAYS_LAST_PHONE_CHANGE": stats,
    "DAYS_LAST_DUE_1ST_VERSION-d-app_DAYS_LAST_PHONE_CHANGE": stats,
    "DAYS_LAST_DUE-s-app_DAYS_BIRTH": stats,
    "DAYS_LAST_DUE-d-app_DAYS_BIRTH": stats,
    "DAYS_LAST_DUE-s-app_DAYS_EMPLOYED": stats,
    "DAYS_LAST_DUE-d-app_DAYS_EMPLOYED": stats,
    "DAYS_LAST_DUE-s-app_DAYS_REGISTRATION": stats,
    "DAYS_LAST_DUE-d-app_DAYS_REGISTRATION": stats,
    "DAYS_LAST_DUE-s-app_DAYS_ID_PUBLISH": stats,
    "DAYS_LAST_DUE-d-app_DAYS_ID_PUBLISH": stats,
    "DAYS_LAST_DUE-s-app_DAYS_LAST_PHONE_CHANGE": stats,
    "DAYS_LAST_DUE-d-app_DAYS_LAST_PHONE_CHANGE": stats,
    "DAYS_TERMINATION-s-app_DAYS_BIRTH": stats,
    "DAYS_TERMINATION-d-app_DAYS_BIRTH": stats,
    "DAYS_TERMINATION-s-app_DAYS_EMPLOYED": stats,
    "DAYS_TERMINATION-d-app_DAYS_EMPLOYED": stats,
    "DAYS_TERMINATION-s-app_DAYS_REGISTRATION": stats,
    "DAYS_TERMINATION-d-app_DAYS_REGISTRATION": stats,
    "DAYS_TERMINATION-s-app_DAYS_ID_PUBLISH": stats,
    "DAYS_TERMINATION-d-app_DAYS_ID_PUBLISH": stats,
    "DAYS_TERMINATION-s-app_DAYS_LAST_PHONE_CHANGE": stats,
    "DAYS_TERMINATION-d-app_DAYS_LAST_PHONE_CHANGE": stats,
    "cnt_paid": stats_sum,
    "cnt_paid_ratio": stats,
    "cnt_unpaid": stats_sum,
    "amt_paid": stats_sum,
    "amt_unpaid": stats_sum,
    "active": stats_sum,
    "completed": stats_sum
}

In [3]:
col_cat = ['NAME_CONTRACT_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'NAME_CASH_LOAN_PURPOSE',
           'NAME_CONTRACT_STATUS', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON',
           'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO',
           'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'NAME_SELLER_INDUSTRY', 'NAME_YIELD_GROUP', "NFLAG_INSURED_ON_APPROVAL",
           'PRODUCT_COMBINATION']
KEY="SK_ID_CURR"

In [4]:
train = utils.get_pickles("train", cols=[KEY])
test = utils.get_pickles("test", cols=[KEY])

In [9]:
def aggregate_cat(prev, args):
    k, v, prefix = args
    
    df = prev.loc[prev[k]==v]
    df = pd.get_dummies(df)
    
    li = [c1 for c1 in df.columns if any(c1.startswith(c2 + '_') for c2 in col_cat)]
    cat_aggregations = {cat: ['mean', 'sum'] for cat in li}
        
    group = df.groupby('SK_ID_CURR')
    df_agg = group.agg({**cat_aggregations})
    df_agg.columns = pd.Index([prefix + e[0] + "_" + e[1] for e in df_agg.columns.tolist()])
    
    group_size = group.size().rename(prefix+'PREV_COUNT')
    df_agg = pd.concat([df_agg, group_size], axis=1)
    df_agg.reset_index(inplace=True)
    
    df_agg.dropna(axis=1, how='all', inplace=True)
    # utils.remove_feature(df_agg, var_limit=0, corr_limit=0.98, sample_size=19999)
    
    return df_agg

def aggregate_num(prev, args, prev_num_aggregations):
    k, v, prefix = args

    df = prev.loc[prev[k] == v]

    df = pd.get_dummies(df, dummy_na=False)

    group = df.groupby('SK_ID_CURR')
    df_agg = group.agg(prev_num_aggregations)
    
    df_agg.columns = pd.Index([prefix + f'{col}_{stat}' for col, stat in df_agg.columns])

    col_std = [c for c in df_agg.columns if c.endswith('_std')]
    col_max = [c for c in df_agg.columns if c.endswith('_max')]

    cols_std = {
        f'{c}_div_mean': df_agg[c] / df_agg[c.replace('_std', '_mean')].replace(0, np.nan)
        for c in col_std
    }
    cols_max = {
        f'{c}_div_min': df_agg[c] / df_agg[c.replace('_max', '_min')].replace(0, np.nan)
        for c in col_max
    }

    if cols_std or cols_max:
        df_agg = pd.concat([df_agg, pd.DataFrame({**cols_std, **cols_max}, index=df_agg.index)], axis=1)

    df_agg.dropna(axis=1, how='all', inplace=True)
    # utils.remove_feature(df_agg, var_limit=0, corr_limit=0.98, sample_size=19999)

    return df_agg

In [6]:
argss = [
    ('NAME_CONTRACT_STATUS', 'Approved', 'approved_'),
    ('NAME_CONTRACT_STATUS', 'Refused', 'refused_'),
    ('NAME_YIELD_GROUP', 'high', 'nyg-high_'),
    ('NAME_YIELD_GROUP', 'middle', 'nyg-middle_'),
    ('NAME_YIELD_GROUP', 'low_normal', 'nyg-low_normal_'),
    ('NAME_YIELD_GROUP', 'low_action', 'nyg-low_action_'),
    ('active',    1, 'active_'),
    ('completed', 1, 'completed_')]

In [7]:
from concurrent.futures import ThreadPoolExecutor

In [8]:
PREF="f101_"
KEY="SK_ID_CURR"

if __name__ == '__main__':
    prev = utils.get_pickles("prev", cols=[KEY] + col_cat + ["active", "completed"])

    def process_item(args):
        return aggregate_cat(prev, args)
    
    li = []
    with ThreadPoolExecutor(max_workers=6) as executor:
        results = executor.map(process_item, argss)
        li.extend(results)
    
    train_merged = train.copy()
    test_merged = test.copy()
    
    for df_agg in list(li):
        train_merged = pd.merge(train_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
        cols_to_drop_train = [col for col in train_merged.columns if col.endswith('_temp')]
        train_merged.drop(columns=cols_to_drop_train, inplace=True)
        train_merged.columns = train_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

        test_merged = pd.merge(test_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
        cols_to_drop_test = [col for col in test_merged.columns if col.endswith('_temp')]
        test_merged.drop(columns=cols_to_drop_test, inplace=True)
        test_merged.columns = test_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

        li.remove(df_agg) # xóa bớt vì nặng
    
    if KEY in train_merged.columns:
        train_merged = train_merged.drop(KEY, axis=1)
    if KEY in test_merged.columns:
        test_merged = test_merged.drop(KEY, axis=1)

    utils.to_feature(train_merged.add_prefix(PREF), name="train")
    utils.to_feature(test_merged.add_prefix(PREF), name="test")
    
if __name__ == '__main__':
    items = list(prev_num_aggregations.items())
    for i in range(0, len(items), 15):
        prev_num_aggregation = dict(items[i:i+15])
        cols = list({KEY, "NAME_CONTRACT_STATUS", "NAME_YIELD_GROUP", "active", "completed"} | set(prev_num_aggregation))
        prev = utils.get_pickles("prev", cols=cols)
        print(prev_num_aggregation , "...\n")

        def process_item(args):
            return aggregate_num(prev, args, prev_num_aggregation)
        
        li = []
        with ThreadPoolExecutor(max_workers=6) as executor:
            results = executor.map(process_item, argss)
            li.extend(results)
        
        train_merged = train.copy()
        test_merged = test.copy()
        
        for df_agg in list(li):
            train_merged = pd.merge(train_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
            cols_to_drop_train = [col for col in train_merged.columns if col.endswith('_temp')]
            train_merged.drop(columns=cols_to_drop_train, inplace=True)
            train_merged.columns = train_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

            test_merged = pd.merge(test_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
            cols_to_drop_test = [col for col in test_merged.columns if col.endswith('_temp')]
            test_merged.drop(columns=cols_to_drop_test, inplace=True)
            test_merged.columns = test_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

            li.remove(df_agg) # xóa bớt vì nặng
        
        if KEY in train_merged.columns:
            train_merged = train_merged.drop(KEY, axis=1)
        if KEY in test_merged.columns:
            test_merged = test_merged.drop(KEY, axis=1)

        utils.to_feature(train_merged.add_prefix(PREF), name="train")
        utils.to_feature(test_merged.add_prefix(PREF), name="test")

d:\Data Science/data/feature/train/f101_approved_NAME_CONTRACT_TYPE_Cash_loans_mean.f
d:\Data Science/data/feature/train/f101_approved_NAME_CONTRACT_TYPE_Cash_loans_sum.f
d:\Data Science/data/feature/train/f101_approved_NAME_CONTRACT_TYPE_Consumer_loans_mean.f
d:\Data Science/data/feature/train/f101_approved_NAME_CONTRACT_TYPE_Consumer_loans_sum.f
d:\Data Science/data/feature/train/f101_approved_NAME_CONTRACT_TYPE_Revolving_loans_mean.f
d:\Data Science/data/feature/train/f101_approved_NAME_CONTRACT_TYPE_Revolving_loans_sum.f
d:\Data Science/data/feature/train/f101_approved_WEEKDAY_APPR_PROCESS_START_FRIDAY_mean.f
d:\Data Science/data/feature/train/f101_approved_WEEKDAY_APPR_PROCESS_START_FRIDAY_sum.f
d:\Data Science/data/feature/train/f101_approved_WEEKDAY_APPR_PROCESS_START_MONDAY_mean.f
d:\Data Science/data/feature/train/f101_approved_WEEKDAY_APPR_PROCESS_START_MONDAY_sum.f
d:\Data Science/data/feature/train/f101_approved_WEEKDAY_APPR_PROCESS_START_SATURDAY_mean.f
d:\Data Science/dat

In [14]:
PREF="f102_"
KEY="SK_ID_CURR"

prev_days = utils.get_pickles("prev", cols=[KEY, "DAYS_DECISION"])
mask = (prev_days['DAYS_DECISION'] >= -365) & (prev_days['DAYS_DECISION'] <= 0)
valid_index = prev_days.index[mask]

if __name__ == '__main__':
    prev = utils.get_pickles("prev", cols=[KEY] + col_cat + ["active", "completed"]).loc[valid_index]

    def process_item(args):
        return aggregate_cat(prev, args)
    
    li = []
    with ThreadPoolExecutor(max_workers=6) as executor:
        results = executor.map(process_item, argss)
        li.extend(results)
    
    train_merged = train.copy()
    test_merged = test.copy()
    
    for df_agg in list(li):
        train_merged = pd.merge(train_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
        cols_to_drop_train = [col for col in train_merged.columns if col.endswith('_temp')]
        train_merged.drop(columns=cols_to_drop_train, inplace=True)
        train_merged.columns = train_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

        test_merged = pd.merge(test_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
        cols_to_drop_test = [col for col in test_merged.columns if col.endswith('_temp')]
        test_merged.drop(columns=cols_to_drop_test, inplace=True)
        test_merged.columns = test_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

        li.remove(df_agg) # xóa bớt vì nặng
    
    if KEY in train_merged.columns:
        train_merged = train_merged.drop(KEY, axis=1)
    if KEY in test_merged.columns:
        test_merged = test_merged.drop(KEY, axis=1)

    utils.to_feature(train_merged.add_prefix(PREF), name="train")
    utils.to_feature(test_merged.add_prefix(PREF), name="test")
    
if __name__ == '__main__':
    items = list(prev_num_aggregations.items())
    for i in range(0, len(items), 15):
        prev_num_aggregation = dict(items[i:i+15])
        cols = list({KEY, "NAME_CONTRACT_STATUS", "NAME_YIELD_GROUP", "active", "completed"} | set(prev_num_aggregation))
        prev = utils.get_pickles("prev", cols=cols).loc[valid_index]
        print(prev_num_aggregation , "...\n")

        def process_item(args):
            return aggregate_num(prev, args, prev_num_aggregation)
        
        li = []
        with ThreadPoolExecutor(max_workers=6) as executor:
            results = executor.map(process_item, argss)
            li.extend(results)
        
        train_merged = train.copy()
        test_merged = test.copy()
        
        for df_agg in list(li):
            train_merged = pd.merge(train_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
            cols_to_drop_train = [col for col in train_merged.columns if col.endswith('_temp')]
            train_merged.drop(columns=cols_to_drop_train, inplace=True)
            train_merged.columns = train_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

            test_merged = pd.merge(test_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
            cols_to_drop_test = [col for col in test_merged.columns if col.endswith('_temp')]
            test_merged.drop(columns=cols_to_drop_test, inplace=True)
            test_merged.columns = test_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

            li.remove(df_agg) # xóa bớt vì nặng
        
        if KEY in train_merged.columns:
            train_merged = train_merged.drop(KEY, axis=1)
        if KEY in test_merged.columns:
            test_merged = test_merged.drop(KEY, axis=1)

        utils.to_feature(train_merged.add_prefix(PREF), name="train")
        utils.to_feature(test_merged.add_prefix(PREF), name="test")

d:\Data Science/data/feature/train/f102_approved_NAME_CONTRACT_TYPE_Cash_loans_mean.f
d:\Data Science/data/feature/train/f102_approved_NAME_CONTRACT_TYPE_Cash_loans_sum.f
d:\Data Science/data/feature/train/f102_approved_NAME_CONTRACT_TYPE_Consumer_loans_mean.f
d:\Data Science/data/feature/train/f102_approved_NAME_CONTRACT_TYPE_Consumer_loans_sum.f
d:\Data Science/data/feature/train/f102_approved_NAME_CONTRACT_TYPE_Revolving_loans_mean.f
d:\Data Science/data/feature/train/f102_approved_NAME_CONTRACT_TYPE_Revolving_loans_sum.f
d:\Data Science/data/feature/train/f102_approved_WEEKDAY_APPR_PROCESS_START_FRIDAY_mean.f
d:\Data Science/data/feature/train/f102_approved_WEEKDAY_APPR_PROCESS_START_FRIDAY_sum.f
d:\Data Science/data/feature/train/f102_approved_WEEKDAY_APPR_PROCESS_START_MONDAY_mean.f
d:\Data Science/data/feature/train/f102_approved_WEEKDAY_APPR_PROCESS_START_MONDAY_sum.f
d:\Data Science/data/feature/train/f102_approved_WEEKDAY_APPR_PROCESS_START_SATURDAY_mean.f
d:\Data Science/dat

In [15]:
PREF="f103_"
KEY="SK_ID_CURR"

# lấy trong 2 năm -> 1 năm 

prev_days = utils.get_pickles("prev", cols=[KEY, "DAYS_DECISION"])
mask = (prev_days['DAYS_DECISION'] >= -365*2) & (prev_days['DAYS_DECISION'] <= -365*1)
valid_index = prev_days.index[mask]

if __name__ == '__main__':
    prev = utils.get_pickles("prev", cols=[KEY] + col_cat + ["active", "completed"]).loc[valid_index]

    def process_item(args):
        return aggregate_cat(prev, args)
    
    li = []
    with ThreadPoolExecutor(max_workers=6) as executor:
        results = executor.map(process_item, argss)
        li.extend(results)
    
    train_merged = train.copy()
    test_merged = test.copy()
    
    for df_agg in list(li):
        train_merged = pd.merge(train_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
        cols_to_drop_train = [col for col in train_merged.columns if col.endswith('_temp')]
        train_merged.drop(columns=cols_to_drop_train, inplace=True)
        train_merged.columns = train_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

        test_merged = pd.merge(test_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
        cols_to_drop_test = [col for col in test_merged.columns if col.endswith('_temp')]
        test_merged.drop(columns=cols_to_drop_test, inplace=True)
        test_merged.columns = test_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

        li.remove(df_agg) # xóa bớt vì nặng
    
    if KEY in train_merged.columns:
        train_merged = train_merged.drop(KEY, axis=1)
    if KEY in test_merged.columns:
        test_merged = test_merged.drop(KEY, axis=1)

    utils.to_feature(train_merged.add_prefix(PREF), name="train")
    utils.to_feature(test_merged.add_prefix(PREF), name="test")
    
if __name__ == '__main__':
    items = list(prev_num_aggregations.items())
    for i in range(0, len(items), 15):
        prev_num_aggregation = dict(items[i:i+15])
        cols = list({KEY, "NAME_CONTRACT_STATUS", "NAME_YIELD_GROUP", "active", "completed"} | set(prev_num_aggregation))
        prev = utils.get_pickles("prev", cols=cols).loc[valid_index]
        print(prev_num_aggregation , "...\n")

        def process_item(args):
            return aggregate_num(prev, args, prev_num_aggregation)
        
        li = []
        with ThreadPoolExecutor(max_workers=6) as executor:
            results = executor.map(process_item, argss)
            li.extend(results)
        
        train_merged = train.copy()
        test_merged = test.copy()
        
        for df_agg in list(li):
            train_merged = pd.merge(train_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
            cols_to_drop_train = [col for col in train_merged.columns if col.endswith('_temp')]
            train_merged.drop(columns=cols_to_drop_train, inplace=True)
            train_merged.columns = train_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

            test_merged = pd.merge(test_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
            cols_to_drop_test = [col for col in test_merged.columns if col.endswith('_temp')]
            test_merged.drop(columns=cols_to_drop_test, inplace=True)
            test_merged.columns = test_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

            li.remove(df_agg) # xóa bớt vì nặng
        
        if KEY in train_merged.columns:
            train_merged = train_merged.drop(KEY, axis=1)
        if KEY in test_merged.columns:
            test_merged = test_merged.drop(KEY, axis=1)

        utils.to_feature(train_merged.add_prefix(PREF), name="train")
        utils.to_feature(test_merged.add_prefix(PREF), name="test")

d:\Data Science/data/feature/train/f103_approved_NAME_CONTRACT_TYPE_Cash_loans_mean.f
d:\Data Science/data/feature/train/f103_approved_NAME_CONTRACT_TYPE_Cash_loans_sum.f
d:\Data Science/data/feature/train/f103_approved_NAME_CONTRACT_TYPE_Consumer_loans_mean.f
d:\Data Science/data/feature/train/f103_approved_NAME_CONTRACT_TYPE_Consumer_loans_sum.f
d:\Data Science/data/feature/train/f103_approved_NAME_CONTRACT_TYPE_Revolving_loans_mean.f
d:\Data Science/data/feature/train/f103_approved_NAME_CONTRACT_TYPE_Revolving_loans_sum.f
d:\Data Science/data/feature/train/f103_approved_WEEKDAY_APPR_PROCESS_START_FRIDAY_mean.f
d:\Data Science/data/feature/train/f103_approved_WEEKDAY_APPR_PROCESS_START_FRIDAY_sum.f
d:\Data Science/data/feature/train/f103_approved_WEEKDAY_APPR_PROCESS_START_MONDAY_mean.f
d:\Data Science/data/feature/train/f103_approved_WEEKDAY_APPR_PROCESS_START_MONDAY_sum.f
d:\Data Science/data/feature/train/f103_approved_WEEKDAY_APPR_PROCESS_START_SATURDAY_mean.f
d:\Data Science/dat

In [16]:
PREF="f104_"
KEY="SK_ID_CURR"

# lấy trong 2 năm -> 1 năm 

prev_days = utils.get_pickles("prev", cols=[KEY, "DAYS_DECISION"])
mask = (prev_days['DAYS_DECISION'] >= -365*3) & (prev_days['DAYS_DECISION'] <= -365*2)
valid_index = prev_days.index[mask]

if __name__ == '__main__':
    prev = utils.get_pickles("prev", cols=[KEY] + col_cat + ["active", "completed"]).loc[valid_index]

    def process_item(args):
        return aggregate_cat(prev, args)
    
    li = []
    with ThreadPoolExecutor(max_workers=6) as executor:
        results = executor.map(process_item, argss)
        li.extend(results)
    
    train_merged = train.copy()
    test_merged = test.copy()
    
    for df_agg in list(li):
        train_merged = pd.merge(train_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
        cols_to_drop_train = [col for col in train_merged.columns if col.endswith('_temp')]
        train_merged.drop(columns=cols_to_drop_train, inplace=True)
        train_merged.columns = train_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

        test_merged = pd.merge(test_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
        cols_to_drop_test = [col for col in test_merged.columns if col.endswith('_temp')]
        test_merged.drop(columns=cols_to_drop_test, inplace=True)
        test_merged.columns = test_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

        li.remove(df_agg) # xóa bớt vì nặng
    
    if KEY in train_merged.columns:
        train_merged = train_merged.drop(KEY, axis=1)
    if KEY in test_merged.columns:
        test_merged = test_merged.drop(KEY, axis=1)

    utils.to_feature(train_merged.add_prefix(PREF), name="train")
    utils.to_feature(test_merged.add_prefix(PREF), name="test")
    
if __name__ == '__main__':
    items = list(prev_num_aggregations.items())
    for i in range(0, len(items), 15):
        prev_num_aggregation = dict(items[i:i+15])
        cols = list({KEY, "NAME_CONTRACT_STATUS", "NAME_YIELD_GROUP", "active", "completed"} | set(prev_num_aggregation))
        prev = utils.get_pickles("prev", cols=cols).loc[valid_index]
        print(prev_num_aggregation , "...\n")

        def process_item(args):
            return aggregate_num(prev, args, prev_num_aggregation)
        
        li = []
        with ThreadPoolExecutor(max_workers=6) as executor:
            results = executor.map(process_item, argss)
            li.extend(results)
        
        train_merged = train.copy()
        test_merged = test.copy()
        
        for df_agg in list(li):
            train_merged = pd.merge(train_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
            cols_to_drop_train = [col for col in train_merged.columns if col.endswith('_temp')]
            train_merged.drop(columns=cols_to_drop_train, inplace=True)
            train_merged.columns = train_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

            test_merged = pd.merge(test_merged, df_agg, on=KEY, how='left', suffixes=('', '_temp'))
            cols_to_drop_test = [col for col in test_merged.columns if col.endswith('_temp')]
            test_merged.drop(columns=cols_to_drop_test, inplace=True)
            test_merged.columns = test_merged.columns.str.replace(r'[/\ ]', '_', regex=True)

            li.remove(df_agg) # xóa bớt vì nặng
        
        if KEY in train_merged.columns:
            train_merged = train_merged.drop(KEY, axis=1)
        if KEY in test_merged.columns:
            test_merged = test_merged.drop(KEY, axis=1)

        utils.to_feature(train_merged.add_prefix(PREF), name="train")
        utils.to_feature(test_merged.add_prefix(PREF), name="test")

d:\Data Science/data/feature/train/f104_approved_NAME_CONTRACT_TYPE_Cash_loans_mean.f
d:\Data Science/data/feature/train/f104_approved_NAME_CONTRACT_TYPE_Cash_loans_sum.f
d:\Data Science/data/feature/train/f104_approved_NAME_CONTRACT_TYPE_Consumer_loans_mean.f
d:\Data Science/data/feature/train/f104_approved_NAME_CONTRACT_TYPE_Consumer_loans_sum.f
d:\Data Science/data/feature/train/f104_approved_NAME_CONTRACT_TYPE_Revolving_loans_mean.f
d:\Data Science/data/feature/train/f104_approved_NAME_CONTRACT_TYPE_Revolving_loans_sum.f
d:\Data Science/data/feature/train/f104_approved_WEEKDAY_APPR_PROCESS_START_FRIDAY_mean.f
d:\Data Science/data/feature/train/f104_approved_WEEKDAY_APPR_PROCESS_START_FRIDAY_sum.f
d:\Data Science/data/feature/train/f104_approved_WEEKDAY_APPR_PROCESS_START_MONDAY_mean.f
d:\Data Science/data/feature/train/f104_approved_WEEKDAY_APPR_PROCESS_START_MONDAY_sum.f
d:\Data Science/data/feature/train/f104_approved_WEEKDAY_APPR_PROCESS_START_SATURDAY_mean.f
d:\Data Science/dat